<a href="https://colab.research.google.com/github/gokadagaya/fittrack-project--2601920-/blob/main/Bookstore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**TASK 1: Database Design and Table Creation**

In [47]:
import sqlite3
import pandas as pd

conn= sqlite3.connect('bookstore.db')
cursor=conn.cursor()
cursor.execute

<function Cursor.execute(sql, parameters=(), /)>

Creating Tables

In [48]:
cursor.execute('''
create table if not exists books(
     book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    price REAL NOT NULL,
    stock_quantity INTEGER DEFAULT 0
)
''')

cursor.execute('''
create table if not exists customers(
   customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    city TEXT,
    join_date TEXT
  )
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS Orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER,
    book_id INTEGER,
    quantity INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    total_amount REAL,
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id),
    FOREIGN KEY (book_id) REFERENCES Books(book_id)
)
''')
conn.commit;

  Display Schemas

In [49]:
def show_schema(table_name):
    print(f"\nSchema for {table_name} table:")

    query = f"PRAGMA table_info({table_name})"
    schema_df = pd.read_sql(query, conn)

    print(schema_df)

show_schema("Books")
show_schema("Customers")
show_schema("Orders")


Schema for Books table:
   cid            name     type  notnull dflt_value  pk
0    0         book_id  INTEGER        0       None   1
1    1           title     TEXT        1       None   0
2    2          author     TEXT        1       None   0
3    3           price     REAL        1       None   0
4    4  stock_quantity  INTEGER        0          0   0

Schema for Customers table:
   cid         name     type  notnull dflt_value  pk
0    0  customer_id  INTEGER        0       None   1
1    1         name     TEXT        1       None   0
2    2        email     TEXT        1       None   0
3    3         city     TEXT        0       None   0
4    4    join_date     TEXT        0       None   0

Schema for Orders table:
   cid          name     type  notnull dflt_value  pk
0    0      order_id  INTEGER        0       None   1
1    1   customer_id  INTEGER        0       None   0
2    2       book_id  INTEGER        0       None   0
3    3      quantity  INTEGER        1       None 

**TASK 2: Data Insertion and Querying**

In [50]:
books_data = [
    ('Python Programming', 'John Smith', 599.99, 25),
    ('Data Science Handbook', 'Jane Doe', 899.50, 15),
    ('Machine Learning Basics', 'Alan Turing', 1299.00, 10),
    ('SQL Essentials', 'Edgar Codd', 499.99, 30),
    ('Web Development', 'Tim Berners', 799.00, 20)
]


customers_data = [
    ('Rahul Sharma', 'rahul@email.com', 'Mumbai', '2024-01-15'),
    ('Priya Patel', 'priya@email.com', 'Delhi', '2024-01-20'),
    ('Amit Kumar', 'amit@email.com', 'Bangalore', '2024-02-01'),
    ('Sneha Reddy', 'sneha@email.com', 'Hyderabad', '2024-02-10'),
    ('Vikram Singh', 'vikram@email.com', 'Mumbai', '2024-02-15')
]


orders_data = [
    (1, 1, 2, '2024-03-01', 1199.00),
    (1, 2, 1, '2024-03-02', 899.50),
    (2, 1, 1, '2024-03-03', 599.99),
    (2, 3, 1, '2024-03-05', 1299.00),
    (3, 4, 3, '2024-03-07', 1499.97),
    (4, 2, 1, '2024-03-10', 899.50),
    (5, 5, 2, '2024-03-12', 1598.00)
]


Part-A:Insert Data

In [51]:
cursor.executemany("""
INSERT INTO Books (title, author, price, stock_quantity)
VALUES (?, ?, ?, ?)
""", books_data)

cursor.executemany("""
INSERT INTO Customers (name, email, city, join_date)
VALUES (?, ?, ?, ?)
""", customers_data)

cursor.executemany("""
INSERT INTO Orders (customer_id, book_id, quantity, order_date, total_amount)
VALUES (?, ?, ?, ?, ?)
""", orders_data)

conn.commit()


In [24]:
for table in ["Books", "Customers", "Orders"]:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    print(df)

   book_id                    title       author    price  stock_quantity
0       21       Python Programming   John Smith   599.99              25
1       22    Data Science Handbook     Jane Doe   899.50              15
2       23  Machine Learning Basics  Alan Turing  1299.00              10
3       24           SQL Essentials   Edgar Codd   499.99              30
4       25          Web Development  Tim Berners   799.00              20
   customer_id          name             email       city   join_date
0            6  Rahul Sharma   rahul@email.com     Mumbai  2024-01-15
1            7   Priya Patel   priya@email.com      Delhi  2024-01-20
2            8    Amit Kumar    amit@email.com  Bangalore  2024-02-01
3            9   Sneha Reddy   sneha@email.com  Hyderabad  2024-02-10
4           10  Vikram Singh  vikram@email.com     Mumbai  2024-02-15
   order_id  customer_id  book_id  quantity  order_date  total_amount
0         8            1        1         2  2024-03-01       1199

Part B: Complex Queries

In [52]:
cursor.execute("select * from customers where city='Mumbai'")
cursor.fetchall()


[(1, 'Rahul Sharma', 'rahul@email.com', 'Mumbai', '2024-01-15'),
 (5, 'Vikram Singh', 'vikram@email.com', 'Mumbai', '2024-02-15')]

In [53]:
cursor.execute("select * from books where price > 800 and stock_quantity > 10")
cursor.fetchall()

[(2, 'Data Science Handbook', 'Jane Doe', 899.5, 15)]

In [54]:
cursor.execute("select count(*) from orders")
print("Total orders",cursor.fetchall()[0])

Total orders (7,)


In [55]:
cursor.execute("select customer_id , count(*) as order_count from Orders group by customer_id order by order_count desc")
cursor.fetchone()

(2, 2)

In [56]:
cursor.execute("SELECT SUM(total_amount) FROM Orders")
print("\nTotal Revenue:", cursor.fetchone()[0])


Total Revenue: 7994.96


**TASK 3: Pandas Integration**

Part A: SQL to Pandas

In [57]:
books_df = pd.read_sql("SELECT * FROM Books", conn)
customers_df = pd.read_sql("SELECT * FROM Customers", conn)
orders_df = pd.read_sql("SELECT * FROM Orders", conn)

print(books_df.head(3))
print(customers_df.head(3))
print(orders_df.head(3))

   book_id                    title       author    price  stock_quantity
0        1       Python Programming   John Smith   599.99              25
1        2    Data Science Handbook     Jane Doe   899.50              15
2        3  Machine Learning Basics  Alan Turing  1299.00              10
   customer_id          name            email       city   join_date
0            1  Rahul Sharma  rahul@email.com     Mumbai  2024-01-15
1            2   Priya Patel  priya@email.com      Delhi  2024-01-20
2            3    Amit Kumar   amit@email.com  Bangalore  2024-02-01
   order_id  customer_id  book_id  quantity  order_date  total_amount
0         1            1        1         2  2024-03-01       1199.00
1         2            1        2         1  2024-03-02        899.50
2         3            2        1         1  2024-03-03        599.99


In [59]:
report = orders_df.merge(customers_df, on='customer_id') \
                  .merge(books_df, on='book_id')

report = report[['order_id', 'name', 'city', 'title', 'quantity', 'total_amount']]
report.columns = ['Order ID', 'Customer Name', 'City', 'Book Title', 'Quantity', 'Total Amount']

print(report)

   Order ID Customer Name       City               Book Title  Quantity  \
0         1  Rahul Sharma     Mumbai       Python Programming         2   
1         2  Rahul Sharma     Mumbai    Data Science Handbook         1   
2         3   Priya Patel      Delhi       Python Programming         1   
3         4   Priya Patel      Delhi  Machine Learning Basics         1   
4         5    Amit Kumar  Bangalore           SQL Essentials         3   
5         6   Sneha Reddy  Hyderabad    Data Science Handbook         1   
6         7  Vikram Singh     Mumbai          Web Development         2   

   Total Amount  
0       1199.00  
1        899.50  
2        599.99  
3       1299.00  
4       1499.97  
5        899.50  
6       1598.00  


In [60]:
print("Average Order Value:", orders_df['total_amount'].mean())

print("\nOrders by City:")
print(report['City'].value_counts())

print("\nMost Popular Book:")
print(report['Book Title'].value_counts().idxmax())

Average Order Value: 1142.1371428571429

Orders by City:
City
Mumbai       3
Delhi        2
Bangalore    1
Hyderabad    1
Name: count, dtype: int64

Most Popular Book:
Python Programming


part B: Pandas to SQL

In [61]:
discounts_data = {
    'book_id': [1, 2, 3, 4, 5],
    'discount_percent': [10, 15, 5, 20, 12]
}

discounts_df = pd.DataFrame(discounts_data)

discounts_df.to_sql("Discounts", conn, if_exists="replace", index=False)

print("✓ Discounts table created")

✓ Discounts table created


In [62]:
pd.read_sql("SELECT * FROM Discounts", conn)

,book_id,discount_percent
0,1,10
1,2,15
2,3,5
3,4,20
4,5,12


In [63]:
discount_query = """
SELECT
    b.title,
    b.price AS original_price,
    d.discount_percent,
    ROUND(b.price * (1 - d.discount_percent / 100), 2) AS discounted_price
FROM Books b
JOIN Discounts d
ON b.book_id = d.book_id
"""

pd.read_sql(discount_query, conn)

,title,original_price,discount_percent,discounted_price
0,Python Programming,599.99,10,599.99
1,Data Science Handbook,899.50,15,899.50
2,Machine Learning Basics,1299.00,5,1299.00
3,SQL Essentials,499.99,20,499.99
4,Web Development,799.00,12,799.00


In [64]:
conn.close()